# AW3D30 elevation quickstart

Pull a tiny ALOS World 3D 30 m DSM tile through the **authless** `jaxa-earth` protocol and inspect the GeoTIFF it writes. The fetch uses the official `jaxa.earth` API; the COG write goes through `pyramids`.

> Needs the `[jaxa]` extra (`pip install 'earthlens[jaxa]'`); no credentials.

## Setup

In [ ]:
import tempfile
from pathlib import Path

from earthlens import EarthLens

## Request

Mt. Fuji area, `resolution=1000` pixels per degree (~110 m at the equator). AW3D30 is a static product — the SDK snaps to its single available epoch (2021-02) once a wide enough date window is passed.

In [ ]:
out_dir = Path(tempfile.mkdtemp(prefix='earthlens-jaxa-'))
lens = EarthLens(
    data_source='jaxa',
    variables=['elevation'],
    start='2000-01-01',
    end='2030-12-31',
    lat_lim=[35.35, 35.40],
    lon_lim=[138.70, 138.78],
    resolution=1000.0,
    path=out_dir,
)

## Download

In [ ]:
written = lens.download()
for p in written:
    print(p, '-', p.stat().st_size, 'bytes')

## Inspect the COG

The GeoTIFF is north-up and in EPSG:4326. Pyramids' `Dataset.read_file` round-trips it without any extra setup.

In [ ]:
from pyramids.dataset import Dataset

cog = Dataset.read_file(str(written[0]))
print('EPSG:        ', cog.epsg)
print('geotransform:', cog.geotransform)
print('shape:       ', cog.shape if hasattr(cog, 'shape') else (cog.rows, cog.columns))
arr = cog.read_array()
print('array shape: ', arr.shape)
print('elevation min/max:', float(arr.min()), float(arr.max()))